# 01 – Préparation du dataset de prévision

## Objectif du notebook

Ce notebook prépare le dataset nécessaire au modèle de prévision de consommation.

Le cas d’usage retenu est "la prévision de la consommation" électrique totale journalière.

L’objectif métier est d’aider Néovolt à anticiper la demande afin de :
- mieux préparer les achats d’énergie ;
- réduire les achats d’urgence ;
- anticiper les pics de consommation ;
- améliorer la stabilité du réseau.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parents[1]

DATA_CLEANED_B_DIR = ROOT_DIR / "volet-b-data-analyst" / "data_cleaned"
OUTPUT_DIR = ROOT_DIR / "volet-c-data-scientist" / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CLEANED_B_DIR, OUTPUT_DIR

(WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-b-data-analyst/data_cleaned'),
 WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-c-data-scientist/outputs'))

In [3]:
conso = pd.read_csv(DATA_CLEANED_B_DIR / "consommation_preparee.csv")

print("Dimensions du dataset consommation préparé :", conso.shape)
conso.head()

Dimensions du dataset consommation préparé : (511700, 36)


,id_pdl,date,consommation_kwh,zone,annee,mois,jour,jour_semaine,nom_jour,weekend,...,temp_max_c,consommation_kwh_brute,flag_conso_manquante,flag_conso_negative,flag_conso_aberrante_iqr,consommation_kwh_clean,flag_conso_imputee,degres_jour_chauffage,jour_froid,jour_chaud
0,PDL-000001,2024-01-01,18.14,Coteaux-Ouest,2024,1,1,0,Monday,False,...,6.01,18.14,False,False,False,18.14,False,15.49,True,False
1,PDL-000001,2024-01-02,15.01,Coteaux-Ouest,2024,1,2,1,Tuesday,False,...,10.73,15.01,False,False,False,15.01,False,12.42,True,False
2,PDL-000001,2024-01-03,12.31,Coteaux-Ouest,2024,1,3,2,Wednesday,False,...,11.63,12.31,False,False,False,12.31,False,8.67,False,False
3,PDL-000001,2024-01-04,NaN,Coteaux-Ouest,2024,1,4,3,Thursday,False,...,3.45,NaN,True,False,False,16.58,True,18.60,True,False
4,PDL-000001,2024-01-05,18.79,Coteaux-Ouest,2024,1,5,4,Friday,False,...,6.88,18.79,False,False,False,18.79,False,16.42,True,False


In [4]:
conso["date"] = pd.to_datetime(conso["date"], errors="coerce")

print("Date min :", conso["date"].min())
print("Date max :", conso["date"].max())
print("Nombre de PDL :", conso["id_pdl"].nunique())
print("Nombre de lignes :", len(conso))

Date min : 2024-01-01 00:00:00
Date max : 2025-12-31 00:00:00
Nombre de PDL : 700
Nombre de lignes : 511700


In [5]:
# Agrégation journalière pour le modèle de prévision

dataset_journalier = (
    conso
    .groupby("date", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        temp_moyenne_c=("temp_moyenne_c", "mean"),
        temp_min_c=("temp_min_c", "mean"),
        temp_max_c=("temp_max_c", "mean"),
        degres_jour_chauffage=("degres_jour_chauffage", "mean"),
        nb_releves=("consommation_kwh_clean", "count"),
        part_lignes_imputees=("flag_conso_imputee", "mean")
    )
)

dataset_journalier["part_lignes_imputees_pct"] = dataset_journalier["part_lignes_imputees"] * 100
dataset_journalier = dataset_journalier.drop(columns=["part_lignes_imputees"])

dataset_journalier.head()

,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,degres_jour_chauffage,nb_releves,part_lignes_imputees_pct
0,2024-01-01,21155.675,30.222393,3.889143,0.669786,9.282471,13.110857,700,15.000000
1,2024-01-02,21209.150,30.298786,4.060386,-0.906257,9.993514,12.939614,700,15.142857
2,2024-01-03,20949.730,29.928186,5.193214,1.706614,10.241586,11.806786,700,12.571429
3,2024-01-04,21531.630,30.759471,2.804043,-1.659914,7.487829,14.195957,700,14.714286
4,2024-01-05,21302.615,30.432307,2.830414,-0.612214,8.862557,14.169586,700,15.571429


In [6]:
# Création des variables calendaires

dataset_journalier["annee"] = dataset_journalier["date"].dt.year
dataset_journalier["mois"] = dataset_journalier["date"].dt.month
dataset_journalier["jour"] = dataset_journalier["date"].dt.day
dataset_journalier["jour_semaine"] = dataset_journalier["date"].dt.dayofweek
dataset_journalier["weekend"] = dataset_journalier["jour_semaine"].isin([5, 6]).astype(int)
dataset_journalier["jour_annee"] = dataset_journalier["date"].dt.dayofyear

def definir_saison(mois):
    if mois in [12, 1, 2]:
        return "hiver"
    elif mois in [3, 4, 5]:
        return "printemps"
    elif mois in [6, 7, 8]:
        return "ete"
    elif mois in [9, 10, 11]:
        return "automne"
    return np.nan

dataset_journalier["saison"] = dataset_journalier["mois"].apply(definir_saison)

dataset_journalier.head()

,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,degres_jour_chauffage,nb_releves,part_lignes_imputees_pct,annee,mois,jour,jour_semaine,weekend,jour_annee,saison
0,2024-01-01,21155.675,30.222393,3.889143,0.669786,9.282471,13.110857,700,15.000000,2024,1,1,0,0,1,hiver
1,2024-01-02,21209.150,30.298786,4.060386,-0.906257,9.993514,12.939614,700,15.142857,2024,1,2,1,0,2,hiver
2,2024-01-03,20949.730,29.928186,5.193214,1.706614,10.241586,11.806786,700,12.571429,2024,1,3,2,0,3,hiver
3,2024-01-04,21531.630,30.759471,2.804043,-1.659914,7.487829,14.195957,700,14.714286,2024,1,4,3,0,4,hiver
4,2024-01-05,21302.615,30.432307,2.830414,-0.612214,8.862557,14.169586,700,15.571429,2024,1,5,4,0,5,hiver


In [7]:
# Encodage des saisons

dataset_model = pd.get_dummies(
    dataset_journalier,
    columns=["saison"],
    drop_first=False
)

dataset_model.head()

,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,degres_jour_chauffage,nb_releves,part_lignes_imputees_pct,annee,mois,jour,jour_semaine,weekend,jour_annee,saison_automne,saison_ete,saison_hiver,saison_printemps
0,2024-01-01,21155.675,30.222393,3.889143,0.669786,9.282471,13.110857,700,15.000000,2024,1,1,0,0,1,False,False,True,False
1,2024-01-02,21209.150,30.298786,4.060386,-0.906257,9.993514,12.939614,700,15.142857,2024,1,2,1,0,2,False,False,True,False
2,2024-01-03,20949.730,29.928186,5.193214,1.706614,10.241586,11.806786,700,12.571429,2024,1,3,2,0,3,False,False,True,False
3,2024-01-04,21531.630,30.759471,2.804043,-1.659914,7.487829,14.195957,700,14.714286,2024,1,4,3,0,4,False,False,True,False
4,2024-01-05,21302.615,30.432307,2.830414,-0.612214,8.862557,14.169586,700,15.571429,2024,1,5,4,0,5,False,False,True,False


## Création de variables de retard

Pour un problème de prévision, les valeurs passées de consommation sont souvent très utiles.

Nous créons donc :
- consommation de la veille ;
- consommation de la semaine précédente ;
- moyenne mobile 7 jours ;
- moyenne mobile 14 jours.

In [8]:
# Tri chronologique

dataset_model = dataset_model.sort_values("date").reset_index(drop=True)

# Variables de retard
dataset_model["conso_lag_1"] = dataset_model["consommation_totale_kwh"].shift(1)
dataset_model["conso_lag_7"] = dataset_model["consommation_totale_kwh"].shift(7)

# Moyennes mobiles calculées uniquement avec les jours précédents
dataset_model["conso_rolling_7"] = (
    dataset_model["consommation_totale_kwh"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

dataset_model["conso_rolling_14"] = (
    dataset_model["consommation_totale_kwh"]
    .shift(1)
    .rolling(window=14)
    .mean()
)

dataset_model.head(20)

,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,degres_jour_chauffage,nb_releves,part_lignes_imputees_pct,annee,...,weekend,jour_annee,saison_automne,saison_ete,saison_hiver,saison_printemps,conso_lag_1,conso_lag_7,conso_rolling_7,conso_rolling_14
0,2024-01-01,21155.675,30.222393,3.889143,0.669786,9.282471,13.110857,700,15.000000,2024,...,0,1,False,False,True,False,NaN,NaN,NaN,NaN
1,2024-01-02,21209.150,30.298786,4.060386,-0.906257,9.993514,12.939614,700,15.142857,2024,...,0,2,False,False,True,False,21155.675,NaN,NaN,NaN
2,2024-01-03,20949.730,29.928186,5.193214,1.706614,10.241586,11.806786,700,12.571429,2024,...,0,3,False,False,True,False,21209.150,NaN,NaN,NaN
3,2024-01-04,21531.630,30.759471,2.804043,-1.659914,7.487829,14.195957,700,14.714286,2024,...,0,4,False,False,True,False,20949.730,NaN,NaN,NaN
4,2024-01-05,21302.615,30.432307,2.830414,-0.612214,8.862557,14.169586,700,15.571429,2024,...,0,5,False,False,True,False,21531.630,NaN,NaN,NaN
5,2024-01-06,16711.885,23.874121,4.574129,-0.205243,9.930686,12.425871,700,11.285714,2024,...,1,6,False,False,True,False,21302.615,NaN,NaN,NaN
6,2024-01-07,16287.325,23.267607,4.951843,0.602214,9.218714,12.048157,700,11.428571,2024,...,1,7,False,False,True,False,16711.885,NaN,NaN,NaN
7,2024-01-08,21736.210,31.051729,2.098814,-1.835343,7.991471,14.901186,700,15.857143,2024,...,0,8,False,False,True,False,16287.325,21155.675,19878.287143,NaN
8,2024-01-09,21479.265,30.684664,3.306629,0.122257,8.492800,13.693371,700,14.714286,2024,...,0,9,False,False,True,False,21736.210,21209.150,19961.220714,NaN
9,2024-01-10,21349.290,30.498986,3.482757,0.060429,7.106386,13.517243,700,14.714286,2024,...,0,10,False,False,True,False,21479.265,20949.730,19999.808571,NaN


In [9]:
# Suppression des lignes incomplètes liées aux variables de retard

nb_lignes_avant = len(dataset_model)

dataset_model = dataset_model.dropna().reset_index(drop=True)

nb_lignes_apres = len(dataset_model)

print("Lignes avant suppression :", nb_lignes_avant)
print("Lignes après suppression :", nb_lignes_apres)
print("Lignes supprimées :", nb_lignes_avant - nb_lignes_apres)

dataset_model.head()

Lignes avant suppression : 731
Lignes après suppression : 717
Lignes supprimées : 14


,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,degres_jour_chauffage,nb_releves,part_lignes_imputees_pct,annee,...,weekend,jour_annee,saison_automne,saison_ete,saison_hiver,saison_printemps,conso_lag_1,conso_lag_7,conso_rolling_7,conso_rolling_14
0,2024-01-15,21376.940,30.538486,3.407486,-0.576029,8.599843,13.592514,700,14.571429,2024,...,0,15,False,False,True,False,17120.185,21736.210,20245.401429,20061.844286
1,2024-01-16,21189.220,30.270314,3.733243,-0.348829,8.199386,13.266757,700,15.142857,2024,...,0,16,False,False,True,False,21376.940,21479.265,20194.077143,20077.648929
2,2024-01-17,21965.800,31.379714,1.345471,-1.302400,6.372986,15.654529,700,16.142857,2024,...,0,17,False,False,True,False,21189.220,21349.290,20152.642143,20076.225357
3,2024-01-18,21103.435,30.147764,4.807029,0.669086,9.361643,12.192971,700,14.285714,2024,...,0,18,False,False,True,False,21965.800,21407.125,20240.715000,20148.801786
4,2024-01-19,20888.260,29.840371,4.918629,1.870986,10.353814,12.081371,700,13.714286,2024,...,0,19,False,False,True,False,21103.435,21587.370,20197.330714,20118.216429


In [10]:
# Visualisation de la consommation totale journalière

fig = px.line(
    dataset_model,
    x="date",
    y="consommation_totale_kwh",
    title="Consommation totale journalière – Dataset de prévision",
    labels={
        "date": "Date",
        "consommation_totale_kwh": "Consommation totale journalière (kWh)"
    }
)

fig.show()

In [11]:
# Séparation temporelle train / test

date_limite_test = pd.Timestamp("2025-10-01")

dataset_model["split"] = np.where(
    dataset_model["date"] < date_limite_test,
    "train",
    "test"
)

dataset_model["split"].value_counts()

split
train    625
test      92
Name: count, dtype: int64

In [12]:
# Vérification des périodes train / test

periode_split = (
    dataset_model
    .groupby("split")
    .agg(
        date_min=("date", "min"),
        date_max=("date", "max"),
        nb_lignes=("date", "count"),
        conso_moyenne=("consommation_totale_kwh", "mean")
    )
)

periode_split

,date_min,date_max,nb_lignes,conso_moyenne
split,,,,
test,2025-10-01,2025-12-31,92,17756.840707
train,2024-01-15,2025-09-30,625,16435.777392


In [13]:
# Export du dataset de modélisation

output_path = OUTPUT_DIR / "dataset_forecast_journalier.csv"

dataset_model.to_csv(output_path, index=False, encoding="utf-8")

print("Dataset de prévision exporté vers :", output_path)
print("Dimensions :", dataset_model.shape)

Dataset de prévision exporté vers : c:\Users\Master\Desktop\Mohamed\1CPDA\Examen_S2\ExaS2\neovolt-grid-plus\volet-c-data-scientist\outputs\dataset_forecast_journalier.csv
Dimensions : (717, 24)


In [14]:
# Taille du fichier exporté

taille_mo = output_path.stat().st_size / 1024**2
print(f"Taille du fichier exporté : {taille_mo:.2f} Mo")

Taille du fichier exporté : 0.16 Mo


## Conclusion de la préparation

Le dataset de prévision journalier a été créé avec succès.

Il contient :
- la consommation totale journalière ;
- les variables météo ;
- les variables calendaires ;
- les variables de saison ;
- les variables de retard ;
- les moyennes mobiles ;
- un indicateur de séparation train/test.